In [1]:
from google.cloud import bigquery
import os

PROJECT_ID = os.getenv("GCP_PROJECT_ID")
DATASET_ID = os.getenv("BQ_DATASET_ID")
client = bigquery.Client(project=PROJECT_ID, location="EU")

def run_query(title, sql):
    print(f"\n{'='*60}")
    print(f"  {title}")
    print(f"{'='*60}")
    df = client.query(sql).to_dataframe()
    print(df.to_string(index=False))
    return df

c:\Users\user\Documents\GitHub\tc-sql-Innovaciber\venv\lib\site-packages\google\api_core\_python_version_support.py:273: FutureWarning: You are using a Python version (3.10.11) which Google will stop supporting in new releases of google.api_core once it reaches its end of life (2026-10-04). Please upgrade to the latest Python version, or at least Python 3.11, to continue receiving updates for google.api_core past that date.
  warnings.warn(message, FutureWarning)


In [2]:
# QUERY 1: Ingresos por mes 

q1 = f"""
SELECT
  FORMAT_TIMESTAMP('%Y-%m', o.order_date) AS mes,
  COUNT(DISTINCT o.order_id)              AS num_pedidos,
  ROUND(SUM(oi.unit_price * oi.quantity), 2) AS ingresos_totales
FROM `{PROJECT_ID}.{DATASET_ID}.orders`      AS o
JOIN `{PROJECT_ID}.{DATASET_ID}.order_items` AS oi
  ON o.order_id = oi.order_id
WHERE o.order_status = 'delivered'
GROUP BY mes
ORDER BY mes
"""

df_q1 = run_query("Q1 — Ingresos por mes", q1)


  Q1 — Ingresos por mes
    mes  num_pedidos  ingresos_totales
2025-05          138        1030307.77
2025-06          148        1209111.27
2025-07          169        1341318.69
2025-08          167        1342588.11
2025-09          167        1235659.17
2025-10          197        1439841.03
2025-11          153        1085099.83
2025-12          183        1379598.66
2026-01          152        1051120.40
2026-02          150        1121432.06
2026-03          191        1485925.53
2026-04          139        1086148.09
2026-05           46         306651.13


In [3]:
# QUERY 2: Top 10 productos más vendidos

q2 = f"""
SELECT
  p.product_name                             AS producto,
  p.brand                                    AS marca,
  SUM(oi.quantity)                           AS unidades_vendidas,
  ROUND(SUM(oi.unit_price * oi.quantity), 2) AS revenue_total
FROM `{PROJECT_ID}.{DATASET_ID}.order_items` AS oi
JOIN `{PROJECT_ID}.{DATASET_ID}.products`    AS p
  ON oi.product_id = p.product_id
GROUP BY producto, marca
ORDER BY unidades_vendidas DESC
LIMIT 10
"""

df_q2 = run_query("Q2 — Top 10 productos más vendidos", q2)


  Q2 — Top 10 productos más vendidos
              producto   marca  unidades_vendidas  revenue_total
    Amazon Smart Light  Amazon                645      559980.21
Fitbit Fitness Tracker  Fitbit                623      813827.38
  Seagate External HDD Seagate                590      389075.79
       Dell UltraSharp    Dell                550      454572.26
          HP HD Webcam      HP                539      707907.87
    Sony PlayStation 5    Sony                500     1047887.65
       HP Spectre x360      HP                394      374327.42
       Samsung Odyssey Samsung                388      345320.28
     Apple AirPods Pro   Apple                376      592649.30
        Google Pixel 8  Google                370      311287.46


In [4]:
# QUERY 3: Pedidos y ticket medio por país 
q3 = f"""
SELECT
  cu.country                     AS pais,
  COUNT(DISTINCT cu.customer_id) AS total_clientes,
  COUNT(DISTINCT o.order_id)     AS total_pedidos,
  ROUND(AVG(pay.amount), 2)      AS ticket_medio
FROM `{PROJECT_ID}.{DATASET_ID}.customers` AS cu
LEFT JOIN `{PROJECT_ID}.{DATASET_ID}.orders`   AS o
  ON cu.customer_id = o.customer_id
LEFT JOIN `{PROJECT_ID}.{DATASET_ID}.payments` AS pay
  ON o.order_id = pay.order_id
GROUP BY pais
ORDER BY total_pedidos DESC
"""

df_q3 = run_query("Q3 — Pedidos y ticket medio por país", q3)


  Q3 — Pedidos y ticket medio por país
pais  total_clientes  total_pedidos  ticket_medio
  KG               4             33       7777.25
  PS               6             29       8828.84
  MK               4             26       7118.95
  RU               4             26       8547.04
  SY               5             26       7438.65
  TV               5             25       7885.23
  PG               5             25       7200.62
  LR               4             25       7819.93
  HN               6             25       7303.55
  CG               5             24       7023.81
  YE               6             24       8612.06
  DM               7             24       8618.68
  RW               5             22       6698.75
  TJ               5             22       7303.78
  IQ               6             22       7438.96
  NP               4             21       7399.17
  RS               5             21       6731.48
  KM               6             21       6531.75
  UA      

In [5]:
# QUERY 4: Tiempo medio de entrega por país
q4 = f"""
SELECT
  s.shipping_country AS pais,
  ROUND(AVG(
    TIMESTAMP_DIFF(s.delivery_date, o.order_date, DAY)
  ), 1)              AS dias_entrega_medio,
  COUNT(*)           AS pedidos_entregados
FROM `{PROJECT_ID}.{DATASET_ID}.shipments` AS s
JOIN `{PROJECT_ID}.{DATASET_ID}.orders`    AS o
  ON s.order_id = o.order_id
WHERE s.shipping_status = 'delivered'
GROUP BY pais
ORDER BY dias_entrega_medio ASC
"""

df_q4 = run_query("Q4 — Tiempo medio de entrega por país", q4)


  Q4 — Tiempo medio de entrega por país
pais  dias_entrega_medio  pedidos_entregados
  MN                 1.8                  13
  KI                 1.8                   5
  TW                 2.0                  12
  MA                 2.0                   5
  KH                 2.1                   7
  BY                 2.1                  11
  TV                 2.1                  10
  LY                 2.1                   7
  JM                 2.2                  13
  ET                 2.2                   5
  SD                 2.3                   7
  DM                 2.3                   9
  LR                 2.3                   7
  MV                 2.3                   7
  GQ                 2.4                   8
  SY                 2.4                   7
  PL                 2.4                   8
  AO                 2.5                  10
  CZ                 2.5                  11
  UG                 2.5                  11
  RO          

In [6]:
# QUERY 5: Margen de beneficio por categoría 

q5 = f"""
SELECT
  cat.name                                                    AS categoria,
  ROUND(SUM(oi.unit_price * oi.quantity), 2)                  AS ingresos,
  ROUND(SUM(p.cost_price  * oi.quantity), 2)                  AS coste,
  ROUND(SUM((oi.unit_price - p.cost_price) * oi.quantity), 2) AS margen_bruto,
  ROUND(
    100.0 * SUM((oi.unit_price - p.cost_price) * oi.quantity)
    / NULLIF(SUM(oi.unit_price * oi.quantity), 0)
  , 2)                                                        AS margen_pct
FROM `{PROJECT_ID}.{DATASET_ID}.order_items` AS oi
JOIN `{PROJECT_ID}.{DATASET_ID}.products`    AS p
  ON oi.product_id = p.product_id
JOIN `{PROJECT_ID}.{DATASET_ID}.categories`  AS cat
  ON p.category_id = cat.category_id
GROUP BY categoria
ORDER BY margen_pct DESC
"""

df_q5 = run_query("Q5 — Margen de beneficio por categoría", q5)


  Q5 — Margen de beneficio por categoría
  categoria   ingresos      coste  margen_bruto  margen_pct
      Audio 1254638.64  752754.52     501884.12       40.00
    Tablets 2253664.71 1403097.15     850567.56       37.74
     Gaming 1332214.61  882284.34     449930.27       33.77
    Storage 1451572.95  962598.74     488974.21       33.69
  Wearables 1955504.24 1301046.58     654457.66       33.47
    Laptops 1270435.42  845524.41     424911.01       33.45
Accessories 1373515.07  927403.73     446111.34       32.48
   Monitors 1238946.74  861071.80     377874.94       30.50
Smartphones 1559816.46 1102629.88     457186.58       29.31
 Smart Home 1424492.90 1064281.81     360211.09       25.29
